In [1]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64" 

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CheckGoldIceberg")
    .master("local[*]")

    # Iceberg package
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.1",
            "org.postgresql:postgresql:42.7.4",
            "org.apache.hadoop:hadoop-aws:3.4.2",
            "software.amazon.awssdk:bundle:2.29.52",
        ])
    )

    # Iceberg extension
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )

    # Iceberg catalog
    .config("spark.sql.catalog.iceberg_catalog", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.iceberg_catalog.catalog-impl", "org.apache.iceberg.jdbc.JdbcCatalog")
    .config("spark.sql.catalog.iceberg_catalog.type", "hadoop")

    # Vì notebook chạy ngoài Docker, dùng localhost nếu Postgres expose port ra host
    .config("spark.sql.catalog.iceberg_catalog.uri", "jdbc:postgresql://localhost:5432/agent4da")
    .config("spark.sql.catalog.iceberg_catalog.jdbc.user", "bigdata")
    .config("spark.sql.catalog.iceberg_catalog.jdbc.password", "#3Bigdata")

    # MinIO warehouse
    .config("spark.sql.catalog.iceberg_catalog.warehouse", "s3a://gold/warehouse/")
    .config("spark.sql.catalog.iceberg_catalog.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")

    # MinIO S3A
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "Admin123!")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    )
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

    .getOrCreate()
)

26/05/17 21:31:44 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [5]:
spark.sql("SHOW NAMESPACES IN iceberg_catalog").show(truncate=False)
spark.sql("SHOW TABLES IN iceberg_catalog.gold").show(truncate=False)

Py4JJavaError: An error occurred while calling o57.sql.
: org.apache.spark.SparkException: Cannot find catalog plugin class for catalog 'iceberg_catalog': org.apache.iceberg.spark.SparkCatalog.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.catalogPluginClassNotFoundForCatalogError(QueryExecutionErrors.scala:1887)
	at org.apache.spark.sql.connector.catalog.Catalogs$.load(Catalogs.scala:71)
	at org.apache.spark.sql.connector.catalog.CatalogManager.$anonfun$catalog$1(CatalogManager.scala:56)
	at scala.collection.mutable.HashMap.getOrElseUpdate(HashMap.scala:469)
	at org.apache.spark.sql.connector.catalog.CatalogManager.catalog(CatalogManager.scala:56)
	at org.apache.spark.sql.connector.catalog.LookupCatalog$CatalogAndNamespace$.unapply(LookupCatalog.scala:86)
	at org.apache.spark.sql.catalyst.analysis.ResolveCatalogs$$anonfun$apply$1.applyOrElse(ResolveCatalogs.scala:104)
	at org.apache.spark.sql.catalyst.analysis.ResolveCatalogs$$anonfun$apply$1.applyOrElse(ResolveCatalogs.scala: